# Experimental noise comparison and reporting

This notebook is post-processing only. It does not train models or modify the model notebooks. It reads existing results from `training_noise_test/`, creates readable training-versus-test noise tables, and generates line graphs. The folder is optional supplementary material; the official clean-test results remain in the model summaries.

The graphs use test sigma on the x-axis and one line per training sigma. No heatmap is generated.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

OUTPUT_ROOTS = {
    'one_stage': ROOT / 'output' / 'one_stage' / 'training_noise_test',
    'two_stage': ROOT / 'output' / 'two_stage' / 'training_noise_test',
    'annulus_router': ROOT / 'output' / 'one_stage_annulus_router' / 'training_noise_test',
    'ablations': ROOT / 'output' / 'experiments' / 'training_noise_test',
}
print('This notebook performs reporting only; no training functions are imported.')

In [ ]:
def collect_family(family, root):
    if not root.exists():
        print(f'Skipping missing family: {family} ({root})')
        return pd.DataFrame()
    rows = []
    for csv_path in sorted(root.rglob('test_noise_robustness.csv')):
        relative = csv_path.relative_to(root).parts
        train_sigma = relative[0].replace('train_sigma', '')
        condition = relative[1] if len(relative) > 2 else ''
        table = pd.read_csv(csv_path)
        table['family'] = family
        table['train_sigma_label'] = train_sigma
        table['condition'] = condition
        rows.append(table)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

family_tables = {family: collect_family(family, root) for family, root in OUTPUT_ROOTS.items()}
for family, table in family_tables.items():
    print(family, table.shape)

In [ ]:
def report_family(family, table, root):
    if table.empty:
        return
    root.mkdir(parents=True, exist_ok=True)
    table.to_csv(root / 'training_vs_test_iou.csv', index=False)
    clean = table[table['noise_sigma'] == 0.0].copy()
    clean.to_csv(root / 'official_clean_test_comparison.csv', index=False)
    figure, axis = plt.subplots(figsize=(11, 6))
    for (train_label, condition), group in table.groupby(['train_sigma_label', 'condition']):
        label = f'train sigma={train_label}' if not condition else f'{condition}, train sigma={train_label}'
        group = group.sort_values('noise_sigma')
        axis.plot(group['noise_sigma'], group['mean_iou'], marker='o', linewidth=2, label=label)
    axis.set_title(f'{family}: training noise versus test noise')
    axis.set_xlabel('Test noise sigma')
    axis.set_ylabel('Mean IoU')
    axis.set_xticks([0.0, 0.001, 0.0025, 0.005, 0.01])
    axis.grid(True, alpha=0.3)
    axis.legend(fontsize=8, ncol=2)
    figure.tight_layout()
    figure.savefig(root / 'training_vs_test_iou.png', dpi=180)
    plt.show()

for family, table in family_tables.items():
    report_family(family, table, OUTPUT_ROOTS[family])

## Reading the graph

The `test_sigma000` point is the official clean-test result for each training condition. The other x-axis points are supplementary test-time robustness results. The entire `training_noise_test/` directory can be omitted from a final report without removing model checkpoints or official clean-test summaries.